# SC-Mamba: Multivariate Benchmark — 17 Datasets (Direction 3)
**Mục tiêu:** Huấn luyện độc lập 17 datasets với Cross-Asset Graph (`num_assets > 1`), auto-derive cấu hình tối ưu cho từng dataset, thu thập Benchmark Bảng Lớn.
- Script: `core/train.py` (đúng, không phải `main.py`)
- Config: `core/config.v_multivariate_{dataset}.yaml` (tạo tự động mỗi lần lặp)
- Checkpoints: `/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints/`

In [ ]:
# Uninstall conflicting Colab defaults
!pip uninstall -y transformers sentence-transformers torch torchvision torchaudio

# Install pinned PyTorch cu121 (must match Mamba wheel builds)
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton==3.0.0

# Pre-built Mamba2 wheels (saves 20-30 min vs compiling from source)
!wget -qO causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!wget -qO mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!pip install causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl
!pip install mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl

# Scientific stack
!pip install yfinance gluonts tqdm utilsforecast pyyaml pandas numpy submitit torchmetrics gpytorch statsforecast properscoring

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 43.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.2 MB/s eta 0:00:00
     ━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ngngsonan/SC-Mamba.git /content/SC-Mamba 2>/dev/null || echo 'Repo already cloned'

%cd /content/SC-Mamba
!git pull


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned
/content/SC-Mamba
Already up to date.


In [ ]:
# @title 02_train_17data_N_uni.py
# =====================================================================
# 🚀 SC-Mamba — Universal Univariate Training (N=1) on 17 Datasets
# =====================================================================
import yaml, os, subprocess

PROJECT_ROOT   = '/content/SC-Mamba'
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

# ── DATASET REGISTRY ────────────────────────────────────────────────
DATASETS = [
    'car_parts_without_missing', 'cif_2016', 'covid_deaths', 'ercot',
    'exchange_rate', 'fred_md', 'hospital', 'm1_monthly', 'm1_quarterly',
    'm3_monthly', 'm3_quarterly', 'nn5_daily_without_missing', 'nn5_weekly',
    'tourism_monthly', 'tourism_quarterly', 'traffic', 'weather'
]

print(f"\n{'='*65}")
print(f"🚀 UNIVERSAL UNIVARIATE TRAINING (N=1) ON {len(DATASETS)} DATASETS")
print(f"{'='*65}")

config = {
    # Core settings
    'seed'                 : 42,
    'debugging'            : False,
    'scaler'               : 'min_max',
    'sin_pos_enc'          : False,
    'sin_pos_const'        : False,
    'encoding_dropout'     : 0.1,
    'handle_constants_model': True,

    'num_assets'           : 1,      # Channel-Independent (N=1)
    'sub_day'              : True,   # Required for sub-daily features (e.g., Traffic/ERCOT)

    # Backbone architecture
    'ssm_config': {
        'bidirectional'         : False,
        'enc_conv'              : True,
        'init_dil_conv'         : True,
        'enc_conv_kernel'       : 5,
        'init_conv_kernel'      : 5,
        'init_conv_max_dilation': 3,
        'global_residual'       : False,
        'in_proj_norm'          : False,
        'initial_gelu_flag'     : True,
        'linear_seq'            : 15,
        'mamba2'                : True,
        'norm'                  : True,
        'norm_type'             : 'layernorm',
        'num_encoder_layers'    : 2,
        'd_state'               : 128,
        'residual'              : False,
        'token_embed_len'       : 1024,
        'block_expansion'       : 2,
        'chunk_size'            : 256,
        'headdim'               : 128,
    },

    # Training schedule
    'num_epochs'           : 30,
    'training_rounds'      : 200,
    'validation_rounds'    : 50,
    'real_test_interval'   : 5,

    # Learning rate
    'lr_scheduler'         : 'cosine',
    'initial_lr'           : 5e-5,
    'learning_rate'        : 1e-7,
    't_max'                : -1,

    # Sequence & Prediction
    'context_len'          : 256,
    'min_seq_len'          : 30,
    'max_seq_len'          : 256,
    'pred_len'             : 60,
    'pred_len_min'         : 10,
    'pred_len_sample'      : True,
    'batch_size'           : 64,
    'no_pos_enc'           : False,

    # Loss
    'loss'                 : 'nll',
    'beta_kl'              : 0.1,
    'multipoint'           : True,
    'sample_multi_pred'    : 0.5,
    'diag_prints'          : True,
    'nll_detach'           : True,

    # Checkpoint
    'model_prefix'         : CHECKPOINT_DIR,
    'wandb'                : False,
    'continue_training'    : False,
    'version'              : 'v_17data_N_uni',

    # Dataset targets
    'real_train_datasets'  : DATASETS,
    'real_test_datasets'   : ['exchange_rate', 'fred_md', 'm1_monthly'],

    # Synthetic prior settings
    'prior_config': {
        'prior_mix_frac'       : 0.7,   # 70% Synthetic GP + 30% Real
        'curriculum_learning'  : False,
        'mixup_prob'           : 0.0,
        'mixup_series'         : 4,
        'damp_and_spike'       : True,
        'damping_noise_ratio'  : 0.05,
        'spike_noise_ratio'    : 0.05,
        'spike_signal_ratio'   : 0.05,
        'spike_batch_ratio'    : 0.05,
        'fp_options': {
            'linear_random_walk_frac': 0,
            'seasonal_only'  : False,
            'scale_noise'    : [0.6, 0.3],
            'trend_exp'      : False,
            'harmonic_scale_ratio': 0.4,
            'harmonic_rate'  : 0.75,
            'trend_additional': True,
            'transition_ratio': 0.0,
        },
        'gp_prior_config': {
            'max_kernels'            : 6,
            'likelihood_noise_level' : 0.4,
            'noise_level'            : 'random',
            'use_original_gp'        : False,
            'gaussians_periodic'     : True,
            'peak_spike_ratio'       : 0.1,
            'subfreq_ratio'          : 0.2,
            'periods_per_freq'       : 0.5,
            'gaussian_sampling_ratio': 0.2,
            'kernel_periods'         : [4, 5, 7, 21, 24, 30, 60, 120],
            'max_period_ratio'       : 1.0,
            'kernel_bank': {
                'matern_kernel'         : 1.5,
                'linear_kernel'         : 1,
                'periodic_kernel'       : 5,
                'polynomial_kernel'     : 0,
                'spectral_mixture_kernel': 0,
            },
        },
    },
}

# =====================================================================
# 1. Save YAML Configuration
# =====================================================================
os.makedirs(f'{PROJECT_ROOT}/core', exist_ok=True)
config_path = f'{PROJECT_ROOT}/core/config.{config["version"]}.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print(f"📝 Config saved: {config_path}")

# =====================================================================
# 2. Verify Validation Datasets
# =====================================================================
REAL_VAL_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets')
os.makedirs(REAL_VAL_DIR, exist_ok=True)
MAX_LEN = 512
padded = 'nopad'

missing = []
for ds in DATASETS + config['real_test_datasets']:
    pkl_name = f'{ds}_{padded}_{MAX_LEN}.pkl'
    if not os.path.exists(os.path.join(REAL_VAL_DIR, pkl_name)):
        missing.append(ds)

missing = list(set(missing))

if missing:
    print(f'\n🔄 Generating {len(missing)} missing dataset(s) from GluonTS...')
    subprocess.run(['python', f'{PROJECT_ROOT}/data/scripts/store_real_datasets.py'])

for ds in DATASETS:
    pkl_name = f'{ds}_{padded}_{MAX_LEN}.pkl'
    pkl_path = os.path.join(REAL_VAL_DIR, pkl_name)
    if os.path.exists(pkl_path):
        size_mb = os.path.getsize(pkl_path) / (1024*1024)
        print(f'  ✅ {ds:30s} → {pkl_name} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {ds:30s} → MISSING')

# =====================================================================
# 3. RUN TRAINING
# =====================================================================
print('\n🚀 Starting Univariate Model Training...')
os.environ['SC_MAMBA_DIAG'] = '1'

cmd = ['python', f'{PROJECT_ROOT}/core/train.py', '-c', config_path]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.stdout.close()
rc = proc.wait()

if rc == 0:
    print("\n✅ Universal Univariate Training Complete.")
else:
    print(f"\n❌ Training failed (exit code {rc}). Stopping.")


🚀 UNIVERSAL UNIVARIATE TRAINING (N=1) ON 17 DATASETS
📝 Config saved: /content/SC-Mamba/core/config.v_17data_N_uni.yaml
  ✅ car_parts_without_missing      → car_parts_without_missing_nopad_512.pkl (2.0 MB)
  ✅ cif_2016                       → cif_2016_nopad_512.pkl (0.1 MB)
  ✅ covid_deaths                   → covid_deaths_nopad_512.pkl (0.9 MB)
  ✅ ercot                          → ercot_nopad_512.pkl (0.1 MB)
  ✅ exchange_rate                  → exchange_rate_nopad_512.pkl (0.3 MB)
  ✅ fred_md                        → fred_md_nopad_512.pkl (0.8 MB)
  ✅ hospital                       → hospital_nopad_512.pkl (0.9 MB)
  ✅ m1_monthly                     → m1_monthly_nopad_512.pkl (0.9 MB)
  ✅ m1_quarterly                   → m1_quarterly_nopad_512.pkl (0.1 MB)
  ✅ m3_monthly                     → m3_monthly_nopad_512.pkl (2.6 MB)
  ✅ m3_quarterly                   → m3_quarterly_nopad_512.pkl (0.5 MB)
  ✅ nn5_daily_without_missing      → nn5_daily_without_missing_nopad_512.pkl (0.9 MB)
 

In [ ]:
!ls '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'
# !rm -rf '/content/SC-Mamba/data/real_data_evals/SCMamba_v_17data_N_uni_best_mase'

SCMamba_v_17data_N_uni_best_mase.pth
SCMamba_v_17data_N_uni_best.pth
SCMamba_v_17data_N_uni_Final.pth
SCMamba_v_17data_N_uni.pth
SCMamba_v2_multi_exchange_rate_best_mase.pth
SCMamba_v2_multi_exchange_rate_best.pth
SCMamba_v2_multi_exchange_rate_Final.pth
SCMamba_v2_multi_exchange_rate.pth
SCMamba_v_config06_uni_17data_best_mase.pth
SCMamba_v_config06_uni_17data_best.pth
SCMamba_v_config06_uni_17data_Final.pth
SCMamba_v_config06_uni_17data.pth


In [ ]:
# @title 12_eval_ckp_N_uni.py
import subprocess, yaml, os
import pandas as pd
from pathlib import Path

# =====================================================================
# SCMamba Evaluation Aggregation Script
# Tự động quét cache, chạy eval_real_dataset.py cho các dataset còn thiếu
# và tổng hợp kết quả (MASE, CRPS, NLL...) ra file CSV.
# =====================================================================

# --- Cấu hình Môi trường ---
os.environ["TRITON_F32_DEFAULT"] = "ieee"
# SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
# PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)

# Cập nhật theo config mới từ training script
CONFIG_NAME = 'config.v_17data_N_uni.yaml'
CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'SCMamba_v_17data_N_uni_best.pth')

QUICK_VALIDATE = False
SELECTED_DATASETS = 'all'      # Hoặc truyền list: ['cif_2016', 'nn5_weekly']
PRED_STYLE = 'multipoint'      # SC-Mamba luôn dùng multipoint

# Dành cho Quick Validation
QUICK_KEYS = ['cif_2016', 'm1_quarterly', 'nn5_weekly', 'tourism_quarterly', 'm1_monthly', 'hospital']

# DATASET_CONFIGS: Nguồn chân lý (Tên dataset, chu kỳ dự đoán, tần suất, Tên hiển thị, Lĩnh vực)
DATASET_CONFIGS = [
    ('car_parts_without_missing',      12, 'M', 'Car Parts',        'Retail'),
    ('cif_2016',                       12, 'M', 'CIF 2016',         'Finance/Competition'),
    ('covid_deaths',                   30, 'D', 'Covid Deaths',     'Epidemiology'),
    ('ercot',                          24, 'H', 'ERCOT Load',       'Energy'),
    ('exchange_rate',                  30, 'B', 'Exchange Rate',    'Forex Finance'),
    ('fred_md',                        12, 'M', 'FRED-MD',          'Macroeconomics'),
    ('hospital',                       12, 'M', 'Hospital',         'Healthcare'),
    ('m1_monthly',                     18, 'M', 'M1 Monthly',       'Economics'),
    ('m1_quarterly',                    8, 'Q', 'M1 Quarterly',     'Economics'),
    ('m3_monthly',                     18, 'M', 'M3 Monthly',       'Economics'),
    ('m3_quarterly',                    8, 'Q', 'M3 Quarterly',     'Economics'),
    ('nn5_daily_without_missing',      56, 'D', 'NN5 Daily',        'Banking'),
    ('nn5_weekly',                      8, 'W', 'NN5 Weekly',       'Banking'),
    ('tourism_monthly',                24, 'M', 'Tourism Monthly',  'Tourism'),
    ('tourism_quarterly',               8, 'Q', 'Tourism Quarterly','Tourism'),
    ('traffic',                        24, 'H', 'Traffic',          'Transportation'),
    ('weather',                        30, 'D', 'Weather',          'Meteorology'),
]

# --- Thiết lập Đường dẫn & Tên Cache ---
# Lấy chính xác tên checkpoint làm tên thư mục Cache (Giữ nguyên cả chữ _best_mase)
EVAL_MODEL_NAME = os.path.basename(CHECKPOINT_PATH).replace('.pth', '')
EVAL_DIR = Path(PROJECT_ROOT) / 'data' / 'real_data_evals' / EVAL_MODEL_NAME / PRED_STYLE
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Lọc dataset cần evaluate
if QUICK_VALIDATE:
    selected = [d for d in DATASET_CONFIGS if d[0] in QUICK_KEYS]
elif SELECTED_DATASETS == 'all':
    selected = DATASET_CONFIGS
else:
    selected = [d for d in DATASET_CONFIGS if d[0] in SELECTED_DATASETS]

# --- Kiểm tra Cache hiện tại ---
cached, missing = [], []
for gluonts_key, pred_len, freq, name, domain in selected:
    # 512 là MAX_LENGTH cố định chặn trên contextual string trong source
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'
    if yml_path.exists():
        cached.append(name)
    else:
        missing.append(name)

print(f"{'='*60}")
print(f"📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH")
print(f"{'='*60}")
print(f'Model name : {EVAL_MODEL_NAME}')
print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Config YAML: {CONFIG_NAME}')
print(f'Eval dir   : {EVAL_DIR}')
print(f'Trạng thái : ✅ Cached: {len(cached)} | ❌ Missing: {len(missing)}')

# --- Gọi Subprocess chạy Evaluation cho dataset chưa có ---
if missing:
    print(f'▶ Sẽ chạy Evaluation cho {len(missing)} dataset mới: {missing}\n')
    cmd = [
        'python', f'{PROJECT_ROOT}/core/eval_real_dataset.py',
        '-c', CHECKPOINT_PATH,
        '-o', EVAL_MODEL_NAME,  # Ép thư mục output bằng chính tên Checkpoint
        '-cfg', os.path.join(PROJECT_ROOT, 'core', CONFIG_NAME) # Để tự lấy tag sub_day=True
    ]

    print(f'💻 Executing: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'\n⚠️ LỖI: eval_real_dataset.py bị crash (Mã lỗi {result.returncode})')
        print(f'--- STDERR ---\n{result.stderr}')
    else:
        print(f'\n✅ Evaluation Script chạy xong không gặp lỗi.')
else:
    print('\n✅ Tất cả datasets yêu cầu đều đã có Cache. Bỏ qua bước chạy mô hình.')

# --- Đọc Load Kết quả vào Pandas DataFrame ---
results = []
for gluonts_key, pred_len, freq, name, domain in selected:
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'

    if not yml_path.exists():
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'MISSING', 'MASE': None})
        continue

    with open(yml_path) as f:
        raw = yaml.safe_load(f)

    if not raw:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'EMPTY', 'MASE': None})
        continue

    metrics = next(iter(raw.values()), {})
    if metrics.get('mase') is None:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'PLACEHOLDER', 'MASE': None})
        continue

    mase_val = metrics.get("mase")
    mcrps_val = metrics.get("mcrps", metrics.get("crps_scaled"))

    mase_str = f"{mase_val:.4f}" if isinstance(mase_val, (int, float)) else "?"
    mcrps_str = f"{mcrps_val:.4f}" if isinstance(mcrps_val, (int, float)) else "?"

    results.append({
        'dataset': name, 'key': gluonts_key, 'domain': domain,
        'freq': freq, 'pred_len': pred_len, 'status': 'OK',
        'MASE': mase_val, 'MAE': metrics.get('mae'),
        'RMSE': metrics.get('rmse'), 'sMAPE': metrics.get('smape'),
        'NLL': metrics.get('nll'), 'CRPS': metrics.get('crps'),
        'mCRPS': mcrps_val,
    })
    print(f'✅ [{domain[:15]:15s}] {name[:20]:20s} MASE={mase_str}  mCRPS={mcrps_str}')

# Lập bảng tổng kết
df = pd.DataFrame(results)
csv_name = f'{EVAL_MODEL_NAME}_quick_results.csv' if QUICK_VALIDATE else f'{EVAL_MODEL_NAME}_all_results.csv'
csv_path = os.path.join(PROJECT_ROOT, 'results', csv_name)
os.makedirs(os.path.join(PROJECT_ROOT, 'results'), exist_ok=True)
df.to_csv(csv_path, index=False)

# --- In Tổng kết ---
print(f'\n{"="*60}')
print(f'📊 TỔNG KẾT — {EVAL_MODEL_NAME}')
print(f'{"="*60}')

ok = df[df['status'] == 'OK']
if not ok.empty:
    cols = ['dataset', 'domain', 'freq', 'pred_len']
    for m in ['MASE', 'sMAPE', 'mCRPS', 'CRPS', 'NLL']:
        if m in ok.columns: cols.append(m)

    # Render table standard python (No Jupyter requirement)
    print(ok[cols].round(4).to_string(index=False))

    print(f'\n💎 Avg MASE: {ok["MASE"].mean():.4f}')
    if 'mCRPS' in ok.columns:
        print(f'💎 Avg mCRPS: {ok["mCRPS"].mean():.4f}')

err = df[df['status'] != 'OK']
if not err.empty:
    print(f'\n⚠️ Phát hiện {len(err)} datasets LỖI (Chưa eval được):')
    for _, r in err.iterrows():
        print(f'   [{r["status"]:12s}] {r["dataset"]}')

print(f'\n💾 Đã lưu full bảng biểu tại: {csv_path}')


📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH
Model name : SCMamba_v_17data_N_uni_best
Checkpoint : /content/SC-Mamba/checkpoints/SCMamba_v_17data_N_uni_best.pth
Config YAML: config.v_17data_N_uni.yaml
Eval dir   : /content/SC-Mamba/data/real_data_evals/SCMamba_v_17data_N_uni_best/multipoint
Trạng thái : ✅ Cached: 17 | ❌ Missing: 0

✅ Tất cả datasets yêu cầu đều đã có Cache. Bỏ qua bước chạy mô hình.
✅ [Retail         ] Car Parts            MASE=1.0427  mCRPS=0.3569
✅ [Finance/Competi] CIF 2016             MASE=1.3244  mCRPS=0.0145
✅ [Epidemiology   ] Covid Deaths         MASE=10.5481  mCRPS=0.0516
✅ [Energy         ] ERCOT Load           MASE=4.1125  mCRPS=0.1349
✅ [Forex Finance  ] Exchange Rate        MASE=1.4373  mCRPS=0.0149
✅ [Macroeconomics ] FRED-MD              MASE=0.7732  mCRPS=0.0220
✅ [Healthcare     ] Hospital             MASE=0.8291  mCRPS=0.0208
✅ [Economics      ] M1 Monthly           MASE=1.3474  mCRPS=0.0369
✅ [Economics      ] M1 Quarterly         MASE=1.9298  mCRPS=0.0239
✅ [Economic

## log results


```md
============================================================
📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH
============================================================
Model name : SCMamba_v_17data_N_uni_best_mase
Checkpoint : /content/SC-Mamba/checkpoints/SCMamba_v_17data_N_uni_best_mase.pth
Config YAML: config.v_17data_N_uni.yaml
Eval dir   : /content/SC-Mamba/data/real_data_evals/SCMamba_v_17data_N_uni_best_mase/multipoint
Trạng thái : ✅ Cached: 17 | ❌ Missing: 0

✅ Tất cả datasets yêu cầu đều đã có Cache. Bỏ qua bước chạy mô hình.
✅ [Retail         ] Car Parts            MASE=1.0346  mCRPS=0.3456
✅ [Finance/Competi] CIF 2016             MASE=1.3131  mCRPS=0.0145
✅ [Epidemiology   ] Covid Deaths         MASE=10.2579  mCRPS=0.0470
✅ [Energy         ] ERCOT Load           MASE=4.0107  mCRPS=0.1295
✅ [Forex Finance  ] Exchange Rate        MASE=1.5313  mCRPS=0.0155
✅ [Macroeconomics ] FRED-MD              MASE=0.8008  mCRPS=0.0204
✅ [Healthcare     ] Hospital             MASE=0.8326  mCRPS=0.0206
✅ [Economics      ] M1 Monthly           MASE=1.3473  mCRPS=0.0367
✅ [Economics      ] M1 Quarterly         MASE=1.9373  mCRPS=0.0235
✅ [Economics      ] M3 Monthly           MASE=1.0001  mCRPS=0.2319
✅ [Economics      ] M3 Quarterly         MASE=1.6365  mCRPS=0.2853
✅ [Banking        ] NN5 Daily            MASE=1.4653  mCRPS=0.4749
✅ [Banking        ] NN5 Weekly           MASE=0.9258  mCRPS=0.2597
✅ [Tourism        ] Tourism Monthly      MASE=3.0243  mCRPS=0.1041
✅ [Tourism        ] Tourism Quarterly    MASE=3.1624  mCRPS=0.0274
✅ [Transportation ] Traffic              MASE=2.3926  mCRPS=0.4443
✅ [Meteorology    ] Weather              MASE=0.7348  mCRPS=0.1581

============================================================
📊 TỔNG KẾT — SCMamba_v_17data_N_uni_best_mase
============================================================
          dataset              domain freq  pred_len    MASE  sMAPE  mCRPS         CRPS         NLL
        Car Parts              Retail    M        12  1.0346 0.8947 0.3456       0.4495  13698.2041
         CIF 2016 Finance/Competition    M        12  1.3131 0.1097 0.0145 1912835.7312      8.0659
     Covid Deaths        Epidemiology    D        30 10.2579 0.2502 0.0470     282.1577 524317.9375
       ERCOT Load              Energy    H        24  4.0107 0.0820 0.1295     828.6813      8.1372
    Exchange Rate       Forex Finance    B        30  1.5313 0.0064 0.0155       0.0075     -3.4847
          FRED-MD      Macroeconomics    M        12  0.8008 0.0709 0.0204    2487.4375      4.4080
         Hospital          Healthcare    M        12  0.8326 0.0938 0.0206      17.1561      4.0491
       M1 Monthly           Economics    M        18  1.3473 0.0954 0.0367    1768.8842      5.9127
     M1 Quarterly           Economics    Q         8  1.9373 0.0842 0.0235    1473.1481      5.8516
       M3 Monthly           Economics    M        18  1.0001 0.0778 0.2319     513.6428      7.9826
     M3 Quarterly           Economics    Q         8  1.6365 0.0618 0.2853     494.9613      8.0705
        NN5 Daily             Banking    D        56  1.4653 0.1806 0.4749       4.7407      3.5037
       NN5 Weekly             Banking    W         8  0.9258 0.0565 0.2597      11.0535      4.4241
  Tourism Monthly             Tourism    M        24  3.0243 0.1664 0.1041    3461.0566      8.7218
Tourism Quarterly             Tourism    Q         8  3.1624 0.1359 0.0274   10362.2454      9.4181
          Traffic      Transportation    H        24  2.3926 0.3041 0.4443       0.0244     -1.7971
          Weather         Meteorology    D        30  0.7348 0.3060 0.1581       1.7515   1702.4449

💎 Avg MASE: 2.2004
💎 Avg mCRPS: 0.1552

💾 Đã lưu full bảng biểu tại: /content/SC-Mamba/results/SCMamba_v_17data_N_uni_best_mase_all_results.csv


============================================================
📊 BẮT ĐẦU ĐÁNH GIÁ MÔ HÌNH
============================================================
Model name : SCMamba_v_17data_N_uni_best
Checkpoint : /content/SC-Mamba/checkpoints/SCMamba_v_17data_N_uni_best.pth
Config YAML: config.v_17data_N_uni.yaml
Eval dir   : /content/SC-Mamba/data/real_data_evals/SCMamba_v_17data_N_uni_best/multipoint
Trạng thái : ✅ Cached: 17 | ❌ Missing: 0

✅ Tất cả datasets yêu cầu đều đã có Cache. Bỏ qua bước chạy mô hình.
✅ [Retail         ] Car Parts            MASE=1.0427  mCRPS=0.3569
✅ [Finance/Competi] CIF 2016             MASE=1.3244  mCRPS=0.0145
✅ [Epidemiology   ] Covid Deaths         MASE=10.5481  mCRPS=0.0516
✅ [Energy         ] ERCOT Load           MASE=4.1125  mCRPS=0.1349
✅ [Forex Finance  ] Exchange Rate        MASE=1.4373  mCRPS=0.0149
✅ [Macroeconomics ] FRED-MD              MASE=0.7732  mCRPS=0.0220
✅ [Healthcare     ] Hospital             MASE=0.8291  mCRPS=0.0208
✅ [Economics      ] M1 Monthly           MASE=1.3474  mCRPS=0.0369
✅ [Economics      ] M1 Quarterly         MASE=1.9298  mCRPS=0.0239
✅ [Economics      ] M3 Monthly           MASE=1.0022  mCRPS=0.2343
✅ [Economics      ] M3 Quarterly         MASE=1.6282  mCRPS=0.2880
✅ [Banking        ] NN5 Daily            MASE=1.4477  mCRPS=0.4742
✅ [Banking        ] NN5 Weekly           MASE=0.9301  mCRPS=0.2622
✅ [Tourism        ] Tourism Monthly      MASE=3.0447  mCRPS=0.1060
✅ [Tourism        ] Tourism Quarterly    MASE=3.1868  mCRPS=0.0281
✅ [Transportation ] Traffic              MASE=2.3772  mCRPS=0.4406
✅ [Meteorology    ] Weather              MASE=0.7268  mCRPS=0.1609

============================================================
📊 TỔNG KẾT — SCMamba_v_17data_N_uni_best
============================================================
          dataset              domain freq  pred_len    MASE  sMAPE  mCRPS         CRPS         NLL
        Car Parts              Retail    M        12  1.0427 0.8938 0.3569       0.4643  13698.1104
         CIF 2016 Finance/Competition    M        12  1.3244 0.1126 0.0145 1918704.9570      8.0732
     Covid Deaths        Epidemiology    D        30 10.5481 0.2548 0.0516     310.1511 524317.3750
       ERCOT Load              Energy    H        24  4.1125 0.0842 0.1349     863.6017      8.1807
    Exchange Rate       Forex Finance    B        30  1.4373 0.0060 0.0149       0.0072     -3.4796
          FRED-MD      Macroeconomics    M        12  0.7732 0.0670 0.0220    2677.7449      4.4469
         Hospital          Healthcare    M        12  0.8291 0.0933 0.0208      17.2894      4.0646
       M1 Monthly           Economics    M        18  1.3474 0.0958 0.0369    1777.3253      5.9121
     M1 Quarterly           Economics    Q         8  1.9298 0.0842 0.0239    1500.0453      5.8317
       M3 Monthly           Economics    M        18  1.0022 0.0780 0.2343     518.7831      7.9995
     M3 Quarterly           Economics    Q         8  1.6282 0.0619 0.2880     499.6277      8.0877
        NN5 Daily             Banking    D        56  1.4477 0.1790 0.4742       4.7336      3.5092
       NN5 Weekly             Banking    W         8  0.9301 0.0567 0.2622      11.1612      4.4406
  Tourism Monthly             Tourism    M        24  3.0447 0.1676 0.1060    3526.1473      8.7341
Tourism Quarterly             Tourism    Q         8  3.1868 0.1370 0.0281   10638.5647      9.4433
          Traffic      Transportation    H        24  2.3772 0.3062 0.4406       0.0242     -1.7925
          Weather         Meteorology    D        30  0.7268 0.3086 0.1609       1.7825   1702.4447

💎 Avg MASE: 2.2169
💎 Avg mCRPS: 0.1571

💾 Đã lưu full bảng biểu tại: /content/SC-Mamba/results/SCMamba_v_17data_N_uni_best_all_results.csv


```